In [1]:
#Define dir
dir = "/Users/felipesanchez/Documents/UoM/PhD/Phd Third Year/Chapter Fertility/ModellingF/"

using Turing, MCMCChains, Distributions, RData, CSV, DataFrames, LinearAlgebra, StatsFuns, Random, Serialization

#Importing the exposures (in thousands) and birth counts by education
ef1 = CSV.read(dir * "Data_Created/exfe1.csv", DataFrame) .* 1000
ef2 = CSV.read(dir * "Data_Created/exfe2.csv", DataFrame) .* 1000
ef3 = CSV.read(dir * "Data_Created/exfe3.csv", DataFrame) .* 1000
ef4 = CSV.read(dir * "Data_Created/exfe4.csv", DataFrame) .* 1000

b1 = CSV.read(dir * "Data_Created/fx1.csv", DataFrame)
b2 = CSV.read(dir * "Data_Created/fx2.csv", DataFrame)
b3 = CSV.read(dir * "Data_Created/fx3.csv", DataFrame)
b4 = CSV.read(dir * "Data_Created/fx4.csv", DataFrame)

# The log fertility (lmu) is calculated as the natural logarithm of the ratio of two datasets (e.g., b1/ef1).
#Females
lmuf1 = round.(log.(b1 ./ ef1), digits=6)
lmuf2 = round.(log.(b2 ./ ef2), digits=6)
lmuf3 = round.(log.(b3 ./ ef3), digits=6)
lmuf4 = round.(log.(b4 ./ ef4), digits=6)

# Calculating log exposures for  females
# This is done by applying the natural logarithm to each column in the datasets and rounding to six decimal places.
#Females
logef1 = mapcols(c -> round.(log.(c), digits=6), ef1)
logef2 = mapcols(c -> round.(log.(c), digits=6), ef2)
logef3 = mapcols(c -> round.(log.(c), digits=6), ef3)
logef4 = mapcols(c -> round.(log.(c), digits=6), ef4)

# 'b' array: Combines fertility datasets.
b = [b1, b2, b3, b4]

# 'e' array: Combines population count datasets (exposure data) for females.
e = [ef1, ef2, ef3, ef4]

# 'loge' array: Contains the natural logarithm of the population counts for analytical purposes.
loge = [logef1, logef2, logef3, logef4]

# 'lmu' array: Combines log fertility rates.
lmu = [lmuf1, lmuf2, lmuf3, lmuf4]

#Serialize
serialize(dir * "Data_Created/b.jls", b)
serialize(dir * "Data_Created/loge.jls", loge)
serialize(dir * "Data_Created/lmu.jls", lmu)
# Deserialize
#b = deserialize(dir * "Data_Created/b.jls")
#loge = deserialize(dir * "Data_Created/loge.jls")


In [2]:
@model function model4(birth, loge, h, t)
    #Model 3 Coherent (betas and kapas) Model each gender
    # Verify that all matrices have the same dimensions
    if any([size(birth[i]) != size(loge[i]) for i in 1:4]) || any([size(birth[1]) != size(birth[i]) for i in 2:4])
        error("The sizes of the matrices are different")
    else
        n_ages, n_periods = size(birth[1])
        n_for = h
        n_groups = length(birth)
        # Prior for a 
        a1 ~ MvNormal(zeros(n_ages), 10)
        a2 ~ MvNormal(zeros(n_ages), 10)
        a3 ~ MvNormal(zeros(n_ages), 10)
        a4 ~ MvNormal(zeros(n_ages), 10)
        # Priors for b
        b ~ Dirichlet(ones(n_ages))
        # Priors and random walk for k
        sigma ~ Exponential(0.1)

        # Drift coefficients c1 before threshold and c2 at and after the threshold
        c1 ~ Normal(0, sqrt(10))
        c2 ~ Normal(0, sqrt(10))
                
        # Initialize k vectors
        k = Vector{Float64}(undef, n_periods)
        
        # Prior for k
        k[1] ~ Normal(0, sigma)  # Assuming k starts at 0 for simplicity

        # Generate Random Walk with threshold
        for i in 2:n_periods
            if i < t
                # Drift is c1 before the threshold
                k[i] ~ Normal(c1 + k[i-1], sigma)
            else
                # Drift is c2 at and after the threshold
                k[i] ~ Normal(c2 + k[i-1], sigma)
            end
        end
    
    # Normalizing k (restriction that sum of k is zero)
    sum_k = sum(k)
    for i in 1:n_periods
        k[i] = k[i] - sum_k / n_periods
    end

        #Drift RW
        c ~ Normal(0, sqrt(10))

        #Init k vectors
        k = Vector(undef, n_periods)
        
        #Prior for k
        k[1] ~ Normal(c, sigma)

        #Generate Random Walk
        for i in 2:n_periods
            k[i] ~ Normal(c + k[i-1], sigma)
        end
        #Normalizing k (restriction sum k is zero)
        sum_k = sum(k)
        for i in 1:n_periods
            k[i] = k[i] - sum_k / n_periods
        end
        #Priors to Log mortality
        Sigma1 ~ Exponential(1)
        Sigma2 ~ Exponential(1)
        Sigma3 ~ Exponential(1)
        Sigma4 ~ Exponential(1)
        #Init Matrix to dead count
        b_est1 = Matrix(undef, n_ages, n_periods)
        b_est2 = Matrix(undef, n_ages, n_periods)
        b_est3 = Matrix(undef, n_ages, n_periods)
        b_est4 = Matrix(undef, n_ages, n_periods)
        #Init matrix to log mortality (Quantiles generated)
        logmu1 = Matrix(undef, n_ages, n_periods)
        logmu2 = Matrix(undef, n_ages, n_periods)
        logmu3 = Matrix(undef, n_ages, n_periods)
        logmu4 = Matrix(undef, n_ages, n_periods)
        
        # Likelihood for d1, d2, d3 and d4
        ###################################
        for x in 1:n_ages
            for t in 1:n_periods
                logmu1[x, t] ~ Normal(a1[x] + b[x] * k[t], Sigma1)
                b_est1[x, t] = exp(logmu1[x, t] + loge[1][x, t])
                birth[1][x, t] ~ Poisson(b_est1[x, t])

                logmu2[x, t] ~ Normal(a2[x] + b[x] * k[t], Sigma2)
                b_est2[x, t] = exp(logmu2[x, t] + loge[2][x, t])
                birth[2][x,t] ~ Poisson(b_est2[x, t])
                
                logmu3[x, t] ~ Normal(a3[x] + b[x] * k[t], Sigma3)
                b_est3[x, t] = exp(logmu3[x, t] + loge[3][x, t])
                birth[3][x, t] ~ Poisson(b_est3[x, t])
                
                logmu4[x, t] ~ Normal(a4[x] + b[x] * k[t], Sigma4)
                b_est4[x, t] = exp(logmu4[x, t] + loge[4][x, t])
                birth[4][x, t] ~ Poisson(b_est4[x, t])
            end
        end
        #Generated quantities (Forecast and validation)
        ###############################################
        #Forecasting kappa (kf =kappa forecast ) 
        kf = Vector(undef, n_for)
        
        kf[1] ~ Normal(c2+ k[n_periods], sigma)
        
        # Generate kf values
        for t in 2:n_for
            kf[t] ~ Normal(c2 + kf[t - 1], sigma)
        end
    end
end 

model3 (generic function with 2 methods)

In [ ]:
#Especification Model 3 (Coherent same beta and kappa for each sex)
#Model 3 Coherent (betas and kappas) Model each gender
modelesp3f = model3(b, loge, 5)
#Chain model 3f
chain_model3f = sample(modelesp3f, NUTS(0.65), 10)
chain_model3f = sample(modelesp3f, NUTS(0.65), 1000)
#Serialize
serialize(dir * "Output/chain_model3f.jls", chain_model3f)
#Read serialized data
#chain_model1 = deserialize(dir * "Output/chain_model1.jls")


┌ Info: Found initial step size
└   ϵ = 3.7252902984619143e-10
Sampling: 100%|█████████████████████████████████████████| Time: 0:00:06
┌ Info: Found initial step size
└   ϵ = 1.9073486328125e-7
Sampling:  11%|████▋                                    |  ETA: 9:08:26